# House Price India Dataset — EDA Assignment Solution

This notebook solves the **Exploratory Data Analysis (EDA Basics)** assignment using Pandas.

**Dataset:** `House Price India.csv`  
**Total records in uploaded dataset:** 14,620 rows × 23 columns

> In Google Colab, upload `House Price India.csv` into the same session before running this notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "House Price India.csv"   # Upload this CSV in Colab before running

df = pd.read_csv(DATA_PATH)
df.head()

## Basic Checks

The assignment asks us to load the dataset, use `df.info()`, `df.describe()`, and handle missing values if any.

In [ ]:
print("Dataset shape:", df.shape)

print("\nData Info:")
df.info()

print("\nDescriptive Statistics:")
display(df.describe())

print("\nMissing Values:")
display(df.isna().sum())

### Basic Check Interpretation

- Dataset shape: **14,620 rows × 23 columns**
- Missing values: **0**
- Duplicate rows: **0**
- Data types: **19 integer columns** and **4 float columns**

Since the dataset has **no missing values**, no missing value treatment was required.


## Q1. What is the average, median, and standard deviation of house prices?

In [ ]:
q1 = df["Price"].agg(["mean", "median", "std"])
q1

**Answer**

- Average price: **₹538,932**
- Median price: **₹450,000**
- Standard deviation: **₹367,532**

**Interpretation:** The mean price is higher than the median, which suggests that expensive houses are pulling the average upward.


## Q2. Which number of bedrooms is most common?

In [ ]:
bedroom_counts = df["number of bedrooms"].value_counts()
most_common_bedrooms = df["number of bedrooms"].mode()[0]

print("Most common bedrooms:", most_common_bedrooms)
display(bedroom_counts)

**Answer:** The most common number of bedrooms is **3 bedrooms**.

- Count: **6,612 houses**
- Percentage: **45.23%**

**Interpretation:** Most houses in the dataset are 3-bedroom houses.


## Q3. Check if the price data is skewed.

In [ ]:
price_skewness = df["Price"].skew()
print("Price skewness:", price_skewness)

plt.figure(figsize=(8, 5))
plt.hist(df["Price"], bins=40)
plt.title("Distribution of House Prices")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

**Answer:** Price skewness is **4.27**.

**Interpretation:** The price distribution is **positively/right skewed**. Most houses are lower to mid-priced, while a smaller number of very expensive houses create a long right tail.


## Q4. What is the average price for each number of bedrooms?

In [ ]:
avg_price_by_bedrooms = (
    df.groupby("number of bedrooms")["Price"]
      .agg(["count", "mean", "median"])
      .sort_index()
)

avg_price_by_bedrooms

**Interpretation:** Average price generally increases as bedroom count increases, but bedroom categories with very few records should be interpreted carefully.

## Q5. What is the relationship between living area and price?  
Use: `Area of the house(excluding basement)`

In [ ]:
area_col = "Area of the house(excluding basement)"

area_price_corr = df[[area_col, "Price"]].corr().loc[area_col, "Price"]
print("Correlation:", area_price_corr)

plt.figure(figsize=(8, 5))
plt.scatter(df[area_col], df["Price"], alpha=0.4)
plt.title("Area Without Basement vs Price")
plt.xlabel("Area of the house excluding basement")
plt.ylabel("Price")
plt.show()

**Answer:** The correlation between area without basement and price is **0.615**.

**Interpretation:** There is a moderate-to-strong positive relationship. Houses with larger above-basement area generally have higher prices.


## Q6. Identify anomalies where houses have high prices but low areas.

For this assignment, I define:
- **High price** = Price greater than or equal to the 95th percentile
- **Low area** = Area without basement less than or equal to the 25th percentile


In [ ]:
price_p95 = df["Price"].quantile(0.95)
area_q25 = df[area_col].quantile(0.25)

anomalies = df[
    (df["Price"] >= price_p95) &
    (df[area_col] <= area_q25)
]

print("95th percentile price:", price_p95)
print("25th percentile area without basement:", area_q25)
print("Number of anomalies:", anomalies.shape[0])

anomaly_columns = [
    "id", "number of bedrooms", "number of bathrooms", "number of floors",
    "waterfront present", area_col, "Area of the basement",
    "living area", "Postal Code", "Price"
]

anomalies[anomaly_columns].sort_values("Price", ascending=False)

**Answer:** Using the above rule, **4 anomalies** were found.

- High-price threshold: **₹1,150,000**
- Low-area threshold: **1,200**

**Interpretation:** These houses are expensive despite having low above-basement area. Possible reasons may include waterfront, basement area, premium postal code, views, or data quality issues.


## Q7. Compare average price based on number of floors and houses with/without waterfront.

In [ ]:
floor_waterfront_price = (
    df.groupby(["number of floors", "waterfront present"])["Price"]
      .agg(["count", "mean", "median"])
      .sort_index()
)

floor_waterfront_price

**Interpretation:** Waterfront houses have much higher average prices than non-waterfront houses. For example, overall non-waterfront houses average about **₹530,417**, while waterfront houses average about **₹1,641,902**.


## Q8. Identify the minimum and maximum house price. What does this indicate?

In [ ]:
min_house = df.loc[df["Price"].idxmin()]
max_house = df.loc[df["Price"].idxmax()]

display(min_house[["id", "number of bedrooms", area_col, "living area", "Postal Code", "waterfront present", "Price"]])
display(max_house[["id", "number of bedrooms", area_col, "living area", "Postal Code", "waterfront present", "Price"]])

**Answer**

- Minimum house price: **₹78,000**
- Maximum house price: **₹7,700,000**

**Interpretation:** The wide price range shows strong variation in property values and confirms that expensive outliers exist in the dataset.


## Q9. Which location zipcode/area has the highest average price?

In [ ]:
zipcode_avg_price = (
    df.groupby("Postal Code")["Price"]
      .agg(["count", "mean", "median", "min", "max"])
      .sort_values("mean", ascending=False)
)

zipcode_avg_price.head(10)

**Answer:** Postal Code **122071** has the highest average price.

- Average price: **₹2,348,311**
- Number of houses: **37**

**Interpretation:** Location is a major driver of house price. Postal Code **122071** appears to be the most premium location in this dataset.


## Q10. Final Insights

1. The average house price is **₹538,932**, while the median is **₹450,000**, showing that high-value properties pull the average upward.
2. The price variable is **right-skewed** with skewness of **4.27**, meaning there are expensive outliers.
3. The most common house type is **3 bedrooms**, with **6,612 records**.
4. Area without basement has a positive correlation of **0.615** with price, meaning larger houses usually sell for more.
5. Waterfront houses are significantly more expensive on average than non-waterfront houses.
6. Postal Code **122071** has the highest average property price.
7. There are **4 high-price/low-area anomalies**, which should be reviewed before predictive modeling.
8. For machine learning, price transformation and outlier handling may improve model performance.
